# 02. Data Cleaning

Lanjutan dari `01_eda.ipynb`. Notebook ini membersihkan dataset mentah menjadi
data siap-pakai (`data/processed/cleaned_data.csv`) dengan :

1. **Perbaikan kolom `Weight`** (kerusakan akibat auto-konversi Excel).
2. **Penanganan missing value** pada `Age (Month)` dan `Weight for Height`.
3. **Validasi** rentang nilai Weight (1–30 kg) yang masuk akal untuk balita.

Catatan: fitur turunan (`Weight for Age`, `Z-Score W/A`, `Z-Score H/A`,
`Weight for Height`, `Z-Score W/H`) **tetap dibawa** di file output karena
pemilihan fitur (dropping) dilakukan pada tahap preprocessing selanjutnya.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", None)

In [2]:
RAW_PATH = "../data/raw/Overall Data.xlsx"
PROCESSED_PATH = "../data/processed/cleaned_data.csv"

df = pd.read_excel(RAW_PATH)
print("Shape awal:", df.shape)

Shape awal: (40071, 11)


## 1. Perbaikan Kolom `Weight`

Kolom `Weight` tercampur 3 tipe data akibat auto-konversi Excel:

- `datetime` → nilai desimal asli (misal `2025-02-08` → `8.2`, yaitu `day.month` (format DD.MM, umum di locale Indonesia/Eropa)).
- `int` / `str` numerik → dikonversi ke `float`.

Aturan konversi datetime mengikuti instruksi: `float(f"{day}.{month}")`.

In [3]:
print("Sebelum perbaikan — tipe data per nilai pada kolom Weight:")
print(df["Weight"].map(type).value_counts())

Sebelum perbaikan — tipe data per nilai pada kolom Weight:
Weight
<class 'datetime.datetime'>    20158
<class 'int'>                  10780
<class 'str'>                   9133
Name: count, dtype: int64


In [4]:
def fix_weight_value(x):
    if isinstance(x, datetime):
        return float(f"{x.day}.{x.month}")
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        return float(x.strip())
    return np.nan


df["Weight"] = df["Weight"].apply(fix_weight_value)
print("Dtype Weight setelah konversi:", df["Weight"].dtype)
print("Jumlah NaN baru pada Weight:", df["Weight"].isna().sum())

Dtype Weight setelah konversi: float64
Jumlah NaN baru pada Weight: 0


## Validasi Rentang Weight (1–30 kg)

Setelah konversi, cek apakah semua nilai berada pada rentang masuk akal untuk
berat badan balita (±1–30 kg).

In [5]:
low_mask = df["Weight"] < 1
high_mask = df["Weight"] > 30

print(f"Weight < 1 kg : {low_mask.sum()}")
print(f"Weight > 30 kg: {high_mask.sum()}")
print("Nilai Weight > 30 kg:", sorted(df.loc[high_mask, "Weight"].unique()))

Weight < 1 kg : 0
Weight > 30 kg: 6
Nilai Weight > 30 kg: [np.float64(91.0), np.float64(2800.0), np.float64(2900.0), np.float64(3000.0), np.float64(3085.0), np.float64(3100.0)]


### Penanganan Outlier Weight

Validasi menemukan dua jenis nilai di luar rentang wajar:

1. **Nilai skala gram** (`2800, 2900, 3000, 3085, 3100`) — jelas merupakan berat
   bayi baru lahir yang tercatat dalam satuan **gram** (2,8–3,1 kg adalah berat lahir
   yang normal). Perbaikan: bagi 1000 → `2.8, 2.9, 3.0, 3.085, 3.1` kg.
2. **Nilai `91`** — secara biologis mustahil sebagai berat balita (91 kg) dan tidak
   dapat dipastikan nilai aslinya. Karena hanya 1 baris, baris ini **dihapus** (drop)
   daripada ditebak.

In [6]:
gram_mask = df["Weight"] >= 1000
n_gram = gram_mask.sum()
df.loc[gram_mask, "Weight"] = df.loc[gram_mask, "Weight"] / 1000
print(f"Nilai skala gram yang dikoreksi (dibagi 1000): {n_gram}")
print(df.loc[gram_mask, "Weight"].tolist())

Nilai skala gram yang dikoreksi (dibagi 1000): 5
[3.085, 3.1, 2.9, 3.0, 2.8]


In [7]:
implausible_mask = (df["Weight"] < 1) | (df["Weight"] > 30)
print("Baris dengan Weight di luar 1–30 kg (akan di-drop):", implausible_mask.sum())
print(df.loc[implausible_mask, ["No.", "Gender", "Age (Month)", "Weight", "Height", "Weight for Height"]])
df = df.loc[~implausible_mask].reset_index(drop=True)
print("Shape setelah drop outlier Weight:", df.shape)

Baris dengan Weight di luar 1–30 kg (akan di-drop): 1
         No. Gender  Age (Month)  Weight  Height Weight for Height
26336  26337      F         28.0    91.0    84.0               NaN
Shape setelah drop outlier Weight: (40070, 11)


## 2. Penanganan Missing Value

### Strategi: **drop baris** (bukan imputasi)

**`Age (Month)`** : 2 nilai hilang (0,005% dari 40.071).
Usia adalah fitur numerik kunci, tetapi hanya 2 baris yang terdampak. Mengisi
(imputasi) dengan median/mean akan menyuntikkan nilai sintetis yang tidak berasal
dari pengukuran, sedangkan drop kehilangan data yang dapat diabaikan. → **drop**.

**`Weight for Height`** : 5 nilai hilang.
Kolom ini adalah **fitur turunan** (dihitung dari berat & tinggi) yang sesuai
metodologi justru **akan dibuang saat pemilihan fitur** untuk mencegah data leakage.
Oleh karena itu imputasi tidak memberi manfaat apa pun bagi pemodelan, dan opsi
paling sederhana & tidak bias adalah **drop** 5 baris tersebut.

In [8]:
print("Missing value sebelum drop:")
print(df[["Age (Month)", "Weight for Height"]].isna().sum())

Missing value sebelum drop:
Age (Month)          2
Weight for Height    4
dtype: int64


In [9]:
missing_mask = df["Age (Month)"].isna() | df["Weight for Height"].isna()
n_dropped_missing = missing_mask.sum()
df = df.loc[~missing_mask].reset_index(drop=True)
print(f"Baris di-drop karena missing value: {n_dropped_missing}")
print("Shape setelah drop missing value:", df.shape)

Baris di-drop karena missing value: 6
Shape setelah drop missing value: (40064, 11)


## 3. Validasi Akhir & Simpan

Pastikan tidak ada missing value tersisa dan seluruh `Weight` berada pada rentang
wajar (1–30 kg), lalu simpan ke `data/processed/cleaned_data.csv`.

In [10]:
print("Missing value total tersisa:", int(df.isna().sum().sum()))
print("Rentang Weight akhir:", df["Weight"].min(), "-", df["Weight"].max(), "kg")
print("Weight di luar 1–30 kg:", int(((df["Weight"] < 1) | (df["Weight"] > 30)).sum()))
print("Dtype Weight akhir:", df["Weight"].dtype)
print("Dtype Age (Month) akhir:", df["Age (Month)"].dtype)

Missing value total tersisa: 0
Rentang Weight akhir: 2.0 - 28.5 kg
Weight di luar 1–30 kg: 0
Dtype Weight akhir: float64
Dtype Age (Month) akhir: float64


In [11]:
import os

os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"Tersimpan: {PROCESSED_PATH}")
print("Shape akhir:", df.shape)

Tersimpan: ../data/processed/cleaned_data.csv


Shape akhir: (40064, 11)


## 4. Ringkasan Before / After Cleaning

In [12]:
df_raw = pd.read_excel(RAW_PATH)

summary = pd.DataFrame({
    "Aspek": [
        "Jumlah baris",
        "Jumlah kolom",
        "Missing Age (Month)",
        "Missing Weight for Height",
        "Tipe campuran kolom Weight",
        "Dtype Weight",
        "Rentang Weight (kg)",
    ],
    "Before": [
        len(df_raw),
        df_raw.shape[1],
        int(df_raw["Age (Month)"].isna().sum()),
        int(df_raw["Weight for Height"].isna().sum()),
        "datetime/int/str",
        str(df_raw["Weight"].dtype),
        "tidak valid (datetime, gram, outlier)",
    ],
    "After": [
        len(df),
        df.shape[1],
        int(df["Age (Month)"].isna().sum()),
        int(df["Weight for Height"].isna().sum()),
        "seragam (numerik)",
        str(df["Weight"].dtype),
        f"{df['Weight'].min():.2f} - {df['Weight'].max():.2f}",
    ],
})
summary

,Aspek,Before,After
0,Jumlah baris,40071,40064
1,Jumlah kolom,11,11
2,Missing Age (Month),2,0
3,Missing Weight for Height,5,0
4,Tipe campuran kolom Weight,datetime/int/str,seragam (numerik)
5,Dtype Weight,object,float64
6,Rentang Weight (kg),"tidak valid (datetime, gram, outlier)",2.00 - 28.50


In [13]:
print("Distribusi kelas target setelah cleaning:")
vc = df["Height for Age"].value_counts()
print(vc)
print((vc / len(df) * 100).round(2))

Distribusi kelas target setelah cleaning:
Height for Age
Not Stunted    23864
Stunted        16200
Name: count, dtype: int64
Height for Age
Not Stunted    59.56
Stunted        40.44
Name: count, dtype: float64
